# MLOps Lab 2 – Create and explore the Responsible AI dashboard

**Nakoba Hands-on Lab – Create and explore the Responsible AI dashboard**

Accuracy alone doesn't tell you **where** a model fails, **why** it predicts what it does, or whether it treats data segments differently. The **Responsible AI (RAI) dashboard** answers those questions. You'll build one for a diabetes classification model with a pipeline of built-in RAI components:

```
model + train/test data ──► [RAI constructor] ──► [error analysis] ──┐
                                             └──► [explanations] ───┴──► [gather] ──► dashboard
```

## Before you start

The cell below installs or updates the packages this notebook needs **into the notebook's own kernel**. (Installing with `pip` in the terminal can put them in a different Python environment, which is a common cause of `ModuleNotFoundError`.)

> If the output ends with *"you may need to restart the kernel"*, click **⟳ Restart** in the toolbar, then continue from the **next** cell.

In [ ]:
%pip install --quiet --upgrade azure-ai-ml azure-identity
%pip install --quiet pyarrow pandas

## Prepare the data

The RAI components read **Parquet** files through **MLTable** definitions (already in `data/rai/train-data` and `data/rai/test-data`). This cell converts the CSV files to Parquet.

> The model was trained on 8 features, so the data must contain only those 8 plus the label `Diabetic`. Any ID column (`PatientID`) is dropped here: the original lab broke because of it.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

train_dir = Path("../data/rai/train-data")
test_dir = Path("../data/rai/test-data")

df_training = pd.read_csv(train_dir / "diabetes.csv").drop(columns=["PatientID"], errors="ignore")
df_test = pd.read_csv(test_dir / "diabetes-test.csv").drop(columns=["PatientID"], errors="ignore")

pq.write_table(pa.Table.from_pandas(df_training, preserve_index=False), train_dir / "diabetes-training.parquet", version="1.0")
pq.write_table(pa.Table.from_pandas(df_test, preserve_index=False), test_dir / "diabetes-test.parquet", version="1.0")

print(f"train: {df_training.shape}, test: {df_test.shape}")
df_training.head()

## Connect to your workspace

On a compute instance, the workspace details are already configured, so `from_config()` finds them.

> If a banner asks you to **Authenticate**, click it and sign in, then run the cell again.

In [ ]:
from lab_helpers import get_ml_client, ensure_cluster

ml_client = get_ml_client()
cluster_name = ensure_cluster(ml_client)

## Register the training and test data

Each folder (MLTable + Parquet) becomes a data asset. If it already exists, it's reused.

In [ ]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data

data_version = "1"
input_train_data = "diabetes_train_mltable"
input_test_data = "diabetes_test_mltable"


def get_or_create(name, path, description):
    try:
        asset = ml_client.data.get(name=name, version=data_version)
        print(f"Found data asset {name}:{data_version}")
    except Exception:
        asset = ml_client.data.create_or_update(
            Data(path=str(path), type=AssetTypes.MLTABLE, description=description,
                 name=name, version=data_version))
        print(f"Created data asset {name}:{data_version}")
    return asset


train_data = get_or_create(input_train_data, train_dir, "RAI diabetes training data")
test_data = get_or_create(input_test_data, test_dir, "RAI diabetes test data")

## Connect to the azureml registry

The built-in RAI components live in Microsoft's shared **azureml** registry.

In [ ]:
from azure.ai.ml import MLClient

ml_client_registry = MLClient(
    credential=ml_client._credential,
    subscription_id=ml_client.subscription_id,
    resource_group_name=ml_client.resource_group_name,
    registry_name="azureml",
)
print(ml_client_registry)

## Register the model

A trained MLflow model is in this repo's **model** folder. Register it in the workspace (each run adds a new version; the code uses whichever version it just registered).

In [ ]:
from azure.ai.ml.entities import Model

model = ml_client.models.create_or_update(
    Model(path="../model", type=AssetTypes.MLFLOW_MODEL, name="local-mlflow-diabetes",
          description="Diabetes classification model (MLflow, 8 features)")
)
model_name = model.name
expected_model_id = f"{model.name}:{model.version}"
azureml_model_id = f"azureml:{expected_model_id}"
print("Registered model:", expected_model_id)

## Get the RAI components

All four use the **same** version, so they work together.

In [ ]:
rai_constructor_component = ml_client_registry.components.get(name="rai_tabular_insight_constructor", label="latest")
version = rai_constructor_component.version
print("RAI components version:", version)

rai_erroranalysis_component = ml_client_registry.components.get(name="rai_tabular_erroranalysis", version=version)
rai_explanation_component = ml_client_registry.components.get(name="rai_tabular_explanation", version=version)
rai_gather_component = ml_client_registry.components.get(name="rai_tabular_insight_gather", version=version)

## Build the RAI pipeline

In [ ]:
from azure.ai.ml import Input, dsl


@dsl.pipeline(
    compute=cluster_name,
    description="RAI insights on diabetes data",
    experiment_name=f"RAI_insights_{model_name}",
)
def rai_decision_pipeline(target_column_name, train_data, test_data):
    # 1. Initialise the RAI insights
    create_rai_job = rai_constructor_component(
        title="RAI dashboard diabetes",
        task_type="classification",
        model_info=expected_model_id,
        model_input=Input(type=AssetTypes.MLFLOW_MODEL, path=azureml_model_id),
        train_dataset=train_data,
        test_dataset=test_data,
        target_column_name=target_column_name,
    )
    create_rai_job.set_limits(timeout=300)

    # 2. Error analysis: where does the model make mistakes?
    error_job = rai_erroranalysis_component(
        rai_insights_dashboard=create_rai_job.outputs.rai_insights_dashboard,
    )
    error_job.set_limits(timeout=300)

    # 3. Explanations: which features drive the predictions?
    explanation_job = rai_explanation_component(
        rai_insights_dashboard=create_rai_job.outputs.rai_insights_dashboard,
        comment="add explanation",
    )
    explanation_job.set_limits(timeout=300)

    # 4. Combine everything into one dashboard
    rai_gather_job = rai_gather_component(
        constructor=create_rai_job.outputs.rai_insights_dashboard,
        insight_3=error_job.outputs.error_analysis,
        insight_4=explanation_job.outputs.explanation,
    )
    rai_gather_job.set_limits(timeout=300)
    rai_gather_job.outputs.dashboard.mode = "upload"

    return {"dashboard": rai_gather_job.outputs.dashboard}

## Point the pipeline at the data

In [ ]:
diabetes_train_pq = Input(type="mltable", path=f"azureml:{input_train_data}:{data_version}", mode="download")
diabetes_test_pq = Input(type="mltable", path=f"azureml:{input_test_data}:{data_version}", mode="download")

In [ ]:
import uuid
from azure.ai.ml import Output

insights_pipeline_job = rai_decision_pipeline(
    target_column_name="Diabetic",
    train_data=diabetes_train_pq,
    test_data=diabetes_test_pq,
)

# save the dashboard to a unique folder so it can be downloaded and viewed
rand_path = str(uuid.uuid4())
insights_pipeline_job.outputs.dashboard = Output(
    path=f"azureml://datastores/workspaceblobstore/paths/{rand_path}/dashboard/",
    mode="upload",
    type="uri_folder",
)

## Submit and wait

The pipeline takes about **15–25 minutes**. This cell prints the link straight away, then the status every 30 seconds until it finishes. You can watch the graph in the studio meanwhile.

In [ ]:
import time
from IPython.display import HTML, display


def submit_and_wait(ml_client, pipeline_job):
    created_job = ml_client.jobs.create_or_update(pipeline_job)
    print("Pipeline job can be accessed at:")
    display(HTML('<a href="{0}" target="_blank">{0}</a>'.format(created_job.studio_url)))
    while created_job.status not in ["Completed", "Failed", "Canceled", "NotResponding"]:
        time.sleep(30)
        created_job = ml_client.jobs.get(created_job.name)
        print("Latest status:", created_job.status)
    if created_job.status != "Completed":
        print("\nThe pipeline did not complete. In the studio, open the RED step -> Outputs + logs -> "
              "user_logs/std_log.txt, and see docs/06-troubleshooting.md (T9).")
    return created_job


insights_job = submit_and_wait(ml_client, insights_pipeline_job)

## Explore the dashboard

1. In the studio, left menu **Models** → **local-mlflow-diabetes** → the version you registered → **Responsible AI** tab → click **RAI dashboard diabetes**.
2. At the top, choose your **compute instance** in **Connect to compute** (some charts need it to recalculate).
3. Explore:
   - **Error analysis:** the tree shows which data segments have the most errors (for example, certain *PlasmaGlucose* and *Age* ranges). Click a node to see its error rate.
   - **Model overview:** accuracy and other metrics, also per **cohort** (data segment).
   - **Data analysis:** class balance and feature distributions.
   - **Feature importances:** which features push predictions up or down overall (**Aggregate**) and for one patient (**Individual**).

**Discuss:** are errors spread evenly, or concentrated in one group? What would you do about it (more data for that segment, a different model, a human review step)?